# tiny-vla Part 6: train Diffusion Policy and ACT-without-VAE on Colab

Two `lerobot-train` runs on `lerobot/pusht`, with checkpoints saved to Google Drive:

1. **Diffusion Policy**: 50,000 steps, checkpoint every 10,000, AMP, CUDA.
2. **ACT without the VAE**: `--policy.use_vae=false`, otherwise the same settings as the local
   Part 3 ACT run (`configs/act_pusht.sh`). It's a pure regression baseline, with no latent.

**If Colab disconnects:** reconnect, run the setup cells again, then re-run the training cell. It
finds the latest checkpoint on Drive and resumes from it.

**Why train on local disk and copy to Drive:** `lerobot-train` marks the newest checkpoint with a
`checkpoints/last` *symlink*, and the Google Drive mount doesn't support symlinks. Training straight
into Drive would crash at the first checkpoint. So each run writes to `/content/runs/...`, a
background loop copies new checkpoints to Drive every 2 minutes, and a resume copies them back.

Runtime: **GPU** (Runtime -> Change runtime type -> T4 or better).

## 1. Install the pinned lerobot and check the GPU

In [ ]:
import sys
assert sys.version_info >= (3, 12), f"lerobot 0.6.1 needs Python >= 3.12, this runtime has {sys.version}"
!pip install -q "lerobot[pusht,diffusion,training]==0.6.1"

In [ ]:
import torch, lerobot
print("lerobot", lerobot.__version__, "| torch", torch.__version__, "| CUDA", torch.version.cuda)
if not torch.cuda.is_available():
    raise SystemExit("No CUDA GPU. Switch the runtime to GPU (Runtime -> Change runtime type) and re-run.")
print("GPU:", torch.cuda.get_device_name(0))
assert lerobot.__version__ == "0.6.1", "the rest of this notebook is written against lerobot 0.6.1"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/tiny-vla/part6")   # checkpoints end up here
LOCAL_ROOT = Path("/content/runs")                           # lerobot-train writes here
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive folder:", DRIVE_ROOT)

## 3. Helpers: resume from Drive, keep Drive in sync

- `restore_from_drive(name)` copies a run's checkpoints from Drive back to local disk and rebuilds
  the `last` symlink. It returns the latest checkpoint's `train_config.json`, or None for a fresh run.
- `start_drive_sync(name)` starts a background `rsync` loop (every 120 s) from local disk to Drive.
  It skips the `last` symlink, which Drive can't store.
- `run_training(name, fresh_args)` puts it together: resume if there's a checkpoint, otherwise start
  with `fresh_args`. It streams the log, then does a final sync.

In [ ]:
import subprocess, shutil, time

def checkpoint_dirs(run_dir: Path) -> list[Path]:
    ckpt_root = run_dir / "checkpoints"
    if not ckpt_root.exists():
        return []
    return sorted(p for p in ckpt_root.iterdir()
                  if p.name.isdigit() and (p / "pretrained_model" / "train_config.json").exists()
                  and (p / "training_state").exists())

def restore_from_drive(name: str) -> Path | None:
    drive_run, local_run = DRIVE_ROOT / name, LOCAL_ROOT / name
    if not checkpoint_dirs(drive_run):
        if local_run.exists():
            shutil.rmtree(local_run)   # a crash before the first checkpoint left nothing useful
        return None
    subprocess.run(["rsync", "-a", f"{drive_run}/", f"{local_run}/"], check=True)
    latest = checkpoint_dirs(local_run)[-1]
    last = local_run / "checkpoints" / "last"
    if last.is_symlink() or last.exists():
        last.unlink()
    last.symlink_to(latest.name)   # lerobot expects checkpoints/last -> newest step folder
    return latest / "pretrained_model" / "train_config.json"

def start_drive_sync(name: str) -> subprocess.Popen:
    local_run, drive_run = LOCAL_ROOT / name, DRIVE_ROOT / name
    drive_run.mkdir(parents=True, exist_ok=True)
    loop = (f"while true; do sleep 120; mkdir -p '{local_run}'; "
            f"rsync -a --exclude 'checkpoints/last' '{local_run}/' '{drive_run}/'; done")
    return subprocess.Popen(["bash", "-c", loop])

def final_sync(name: str):
    subprocess.run(["rsync", "-a", "--exclude", "checkpoints/last",
                    f"{LOCAL_ROOT / name}/", f"{DRIVE_ROOT / name}/"], check=True)

def run_training(name: str, fresh_args: list[str]):
    resume_config = restore_from_drive(name)
    if resume_config is not None:
        print(f"Resuming {name} from {resume_config.parent.parent.name}")
        cmd = ["lerobot-train", f"--config_path={resume_config}", "--resume=true"]
    else:
        print(f"Starting {name} from scratch")
        cmd = ["lerobot-train", *fresh_args, f"--output_dir={LOCAL_ROOT / name}"]
    print(" ".join(cmd))
    sync = start_drive_sync(name)
    log_path = DRIVE_ROOT / f"{name}_train.log"
    try:
        with open(log_path, "a") as log:
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in proc.stdout:        # stream to the notebook and to a log file on Drive
                log.write(line)
                if "step:" in line or "Checkpoint" in line or "Error" in line or "Traceback" in line:
                    print(line, end="")
            proc.wait()
    finally:
        sync.terminate()
        final_sync(name)
    if proc.returncode != 0:
        raise RuntimeError(f"lerobot-train exited with {proc.returncode}; see {log_path}")
    print(f"{name} done. Checkpoints: {[p.name for p in checkpoint_dirs(DRIVE_ROOT / name)]}")

## 4. Diffusion Policy: print and save the full config

This builds the same `DiffusionConfig` the training run will use (defaults plus the overrides
passed below) and saves it next to the run's checkpoints on Drive as `diffusion_config.json`. It
goes *next to* the run folder rather than inside it, because `lerobot-train` refuses to start if its
output folder already exists. Every checkpoint also stores the complete run config in
`pretrained_model/train_config.json`.

In [ ]:
import dataclasses, json
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig

DIFFUSION_OVERRIDES = dict(device="cuda", use_amp=True, push_to_hub=False)
diffusion_cfg = DiffusionConfig(**DIFFUSION_OVERRIDES)
diffusion_cfg_dict = json.loads(json.dumps(dataclasses.asdict(diffusion_cfg), default=str))
print(json.dumps(diffusion_cfg_dict, indent=2))
(DRIVE_ROOT / "diffusion_config.json").write_text(json.dumps(diffusion_cfg_dict, indent=2))
print("saved", DRIVE_ROOT / "diffusion_config.json")

## 5. Train Diffusion Policy (50k steps, checkpoint every 10k)

`batch_size=64` matches how the reference `lerobot/diffusion_pusht` was trained (lerobot's default
of 8 is small for diffusion). `env_eval_freq=0` skips sim evaluation during training, because
evaluation happens locally with the same harness for every model. Re-run this cell after a disconnect.

In [ ]:
run_training("diffusion_pusht", [
    "--policy.type=diffusion",
    "--policy.device=cuda",
    "--policy.use_amp=true",
    "--policy.push_to_hub=false",
    "--dataset.repo_id=lerobot/pusht",
    "--env.type=pusht",
    "--batch_size=64",
    "--num_workers=2",
    "--steps=50000",
    "--save_freq=10000",
    "--env_eval_freq=0",
    "--log_freq=200",
    "--wandb.enable=false",
    "--seed=1000",
])

## 6. Train ACT without the VAE (pure regression baseline)

Same settings as the local Part 3 run (`configs/act_pusht.sh`): chunk_size 20, n_action_steps 10,
batch 8, 30k steps, checkpoint every 5k, AMP, seed 1000. The only change is
`--policy.use_vae=false` (`ACTConfig.use_vae` in lerobot 0.6.1). With it off, the CVAE encoder isn't
built, z is always zero, and the loss is plain L1. Re-run this cell after a disconnect.

In [ ]:
run_training("act_novae_pusht", [
    "--policy.type=act",
    "--policy.use_vae=false",
    "--policy.device=cuda",
    "--policy.use_amp=true",
    "--policy.chunk_size=20",
    "--policy.n_action_steps=10",
    "--policy.push_to_hub=false",
    "--dataset.repo_id=lerobot/pusht",
    "--env.type=pusht",
    "--batch_size=8",
    "--num_workers=2",
    "--steps=30000",
    "--save_freq=5000",
    "--env_eval_freq=0",
    "--log_freq=100",
    "--wandb.enable=false",
    "--seed=1000",
])

## 7. Bring the checkpoints home

Download from Drive (`MyDrive/tiny-vla/part6/`) into the repo on the Linux box:

```
outputs/part6/colab/diffusion_pusht/checkpoints/050000/pretrained_model/
outputs/part6/colab/act_novae_pusht/checkpoints/030000/pretrained_model/
```

Only the `pretrained_model/` folders are needed for evaluation. `training_state/` is only for resuming.

In [ ]:
for name in ["diffusion_pusht", "act_novae_pusht"]:
    for ckpt in checkpoint_dirs(DRIVE_ROOT / name):
        size = sum(f.stat().st_size for f in (ckpt / "pretrained_model").rglob("*") if f.is_file()) / 2**20
        print(f"{name}/{ckpt.name}/pretrained_model  {size:.0f} MB")